# TriageLM: Multi-Task Edge-SLM Customer Triage with Hinglish OOD Evaluation

This notebook trains **Qwen2.5-1.5B-Instruct** using 4-bit QLoRA to convert raw, unstructured customer support messages into strict, schema-validated JSON across 5 tasks:
1. **Intent Classification** (Banking & Fintech categories)
2. **Sentiment Detection** (`positive`, `neutral`, `frustrated`, `angry`)
3. **Urgency Scoring** (`low`, `medium`, `high`, `critical`)
4. **Department Routing** (`card_security`, `payments_ops`, `dispute_management`, `account_services`, `general_support`)
5. **Entity Extraction** (`amount`, `currency`, `card_type`, `merchant`, `date`, `transaction_id`)

It also evaluates true **Zero-Shot Out-of-Distribution (OOD)** generalization on unseen **Hinglish (Hindi-English code-mixed)** customer queries.

In [ ]:
# 1. Environment & Dependencies Installation
!pip install -q transformers==4.44.2 peft==0.12.0 trl==0.9.6 bitsandbytes accelerate datasets pydantic rich

In [ ]:
# 2. GPU Verification
!nvidia-smi

In [ ]:
# 3. Pydantic Target Schema
import json
from enum import Enum
from typing import Optional
from pydantic import BaseModel, Field

class SentimentEnum(str, Enum):
    POSITIVE = "positive"
    NEUTRAL = "neutral"
    FRUSTRATED = "frustrated"
    ANGRY = "angry"

class UrgencyEnum(str, Enum):
    LOW = "low"
    MEDIUM = "medium"
    HIGH = "high"
    CRITICAL = "critical"

class RoutingTeamEnum(str, Enum):
    CARD_SECURITY = "card_security"
    PAYMENTS_OPS = "payments_ops"
    DISPUTE_MANAGEMENT = "dispute_management"
    ACCOUNT_SERVICES = "account_services"
    GENERAL_SUPPORT = "general_support"

class ExtractedEntities(BaseModel):
    amount: Optional[str] = None
    currency: Optional[str] = None
    card_type: Optional[str] = None
    merchant: Optional[str] = None
    date: Optional[str] = None
    transaction_id: Optional[str] = None

class TriageOutput(BaseModel):
    intent: str
    sentiment: SentimentEnum
    urgency: UrgencyEnum
    route_to: RoutingTeamEnum
    action_required: bool
    entities: ExtractedEntities = Field(default_factory=ExtractedEntities)

print("Schema initialized successfully!")

In [ ]:
# 4. QLoRA 4-bit Quantization Configuration & Model Loading
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()